# Test Functionality of GestionOt{class}

### Pasos para calificar una actividad.


1. Separar los eventos que tienen alimentador de los que no.
   
   1.1. Separar y calificar aquellos que son de TRANSPORTE, ALIMENTACIÓN, SE LABORA, INFO, se repite en la calificación
   
2. A los eventos que si tienen alimentador.
   
   2.1. Separar aquellos que sabemos que son SAPG, los más fáciles de identificar.

   2.2. Separar aquellos que son de Servicios Ocasionales.

   2.3. Calificar usando la Red Neuronal.

In [1]:

from eerssa import gestionOT
from eerssa import matrizActividades
from pathlib import Path
import pandas as pd

import pickle


#test_path = '/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/'
#test_path = '/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/find_bug'
#test_path = '/home/vlad/OneDrive/01 JEZO/01 ACTIVIDADES DIARIAS DE TRABAJO DE LAS AGENCIAS/2024'
test_path = '/home/vlad/OneDrive/01 JEZO/01 ACTIVIDADES DIARIAS DE TRABAJO DE LAS AGENCIAS/2025/restantes_febrero'
#test_path = '/home/vlad/OneDrive/01 JEZO/01 ACTIVIDADES DIARIAS DE TRABAJO DE LAS AGENCIAS/2025/03 MARZO'


save_dir = '/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/db_test/'

file_prefix = '2025_02_Febrero_v022'

path_obj = save_dir + file_prefix + '_1_object.pkl'
path_pkl = save_dir + file_prefix + '_2_df.pkl'
path_xls = save_dir + file_prefix + '_0_df.xlsx'
path_duk = save_dir + file_prefix + '_3_df.duck'
path_pqt = save_dir + file_prefix + '_4_df.parquet'


list_pdfs = []
for path in Path( test_path ).glob("**/*.pdf"):
  list_pdfs.append( str(path) )
  list_pdfs.sort()


Success!!!


### DASK

Primero creo un cluster locar de computación

In [2]:
from dask.distributed import LocalCluster
client = LocalCluster().get_client()

Luego, con el listado de OT's de ejecutado en primera instancia, genero objetos en los cluster de computación local, 

Ejecuto la función `load_ot()` y guardo los resultados a la misma lista de objetos

In [3]:

futures = [client.submit(gestionOT.GestionOt, file, actor=True) for file in list_pdfs ]
ot_array = [future.result() for future in futures]

ot_cargada = [ot.load_ot() for ot in ot_array]
obj_lists = [future.result() for future in ot_cargada]



Success!!!
Success!!!
Success!!!
Success!!!


🔥

TODO:  Subir a Mongo DB || Mongo Driver

🔥
> TODO
> Imprimir un reporte de las OT que no fue exitoso su conversion a OT. informar las novedades encontradas

🔥
> TODO 
> La siguiente linea de código es posible que no se este ejecutando en paralelo, verificarlo luego

Estoy simultaneamente, generando `matriz` que es un `Pandas.Dataframe` almacenandola en el objeto y en `ot_matrices`

In [6]:
from eerssa import matrizActividades as mt
from importlib import reload
reload(mt)

<module 'eerssa.matrizActividades' from '/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/eerssa/matrizActividades.py'>

In [7]:

ot_matrices = [ mt.ConvertirOT_a_ActividadesCSV(ot) for ot in obj_lists ]
df_total = [ df for df in ot_matrices if df is not None ]

/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/.venv/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MultinomialNB from version 1.4.1.post1 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/.venv/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator CountVectorizer from version 1.4.1.post1 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/.venv/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarni


Error: al intentar calcular el tiempo transcurrido para una actividad . texto de inicio 2025-02-28 12:10:00 texto de fin None



/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/.venv/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MultinomialNB from version 1.4.1.post1 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/.venv/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator CountVectorizer from version 1.4.1.post1 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/.venv/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarni

In [8]:
# Combined full df for exporting to excel

combined_df = pd.concat(df_total, ignore_index=True)

In [9]:

with open( path_obj, 'wb' ) as fp:
  pickle.dump( obj_lists, fp )

combined_df.to_pickle( path_pkl )

In [10]:
import duckdb

# create the table "my_table" from the DataFrame "my_df"
# Note: duckdb.sql connects to the default in-memory database connection
duckdb.sql("CREATE TABLE duck AS SELECT * FROM combined_df")

# insert into the table "my_table" from the DataFrame "my_df"
duckdb.sql("INSERT INTO duck SELECT * FROM combined_df")

## EXCELL Export

Primero importar el dataframe

Segundo Pintar el dataframe

Agrupar por mes

Exportar

POR HACER



In [10]:

df = pd.read_pickle( path_pkl )

df['Cuadrilla'] = df['Cuadrilla'].apply( lambda s: s.split('(')[0] )

df['Cuenta']         = pd.Categorical(df.Cuenta)
df['Dia']            = pd.Categorical(df.Dia)
df['Alimentador']    = pd.Categorical(df.Alimentador)
df['Tipo']           = pd.Categorical(df.Tipo)
df['Actividad']      = pd.Categorical(df.Actividad)
df['Cuadrilla']      = pd.Categorical(df.Cuadrilla)
df['Responsable']    = pd.Categorical(df.Responsable)
df['Vehiculo']       = pd.Categorical(df.Vehiculo)
df['anio_mes']       = df['Fecha'].apply( lambda x: x[:-3] )
df['anio_mes']       = pd.Categorical(df.anio_mes)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 938 entries, 0 to 937
Data columns (total 25 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   Item           938 non-null    int64   
 1   Cuenta         938 non-null    category
 2   Evento         938 non-null    object  
 3   Actividad      938 non-null    category
 4   Alimentador    938 non-null    category
 5   Primario       938 non-null    object  
 6   Desconexion    938 non-null    object  
 7   SIG            938 non-null    object  
 8   Tipo           938 non-null    category
 9   Materiales     938 non-null    object  
 10  Cuadrilla      938 non-null    category
 11  Dia            938 non-null    category
 12  Fecha          938 non-null    object  
 13  InicioEvento   938 non-null    object  
 14  FinEvento      938 non-null    object  
 15  Duracion       938 non-null    int64   
 16  Responsable    938 non-null    category
 17  Colaboradores  938 non-null    int6

In [11]:
df['Actividad'].unique()

['INFO', 'NO PROG', 'LABORA', 'PROG', 'TRANSP', '·', 'ALIMEN']
Categories (7, object): ['ALIMEN', 'INFO', 'LABORA', 'NO PROG', 'PROG', 'TRANSP', '·']

In [12]:
import xlsxwriter

# Create an ExcelWriter object
writer = pd.ExcelWriter( path_xls, engine='xlsxwriter')  


# Group the DataFrame by the categorical column
grouped = df.groupby('Cuadrilla')

# Iterate through groups and write to separate sheets
for category, group_data in grouped:
    group_data.to_excel(writer, sheet_name=str(category), index=False)  

# Save the Excel file
writer.close()


/tmp/ipykernel_6844/1222790251.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = df.groupby('Cuadrilla')


# TESTING

In [ ]:
df[['Item', 'InicioEvento', 'FinEvento' ]]

,Item,InicioEvento,FinEvento
0,1,2025-02-22 00:00:01,2025-02-22 00:00:02
1,2,2025-02-22 12:18:00,2025-02-22 14:29:00
2,5,2025-02-22 00:00:01,2025-02-22 00:00:02
3,1,2025-02-24 14:00:00,2025-02-24 15:00:00
4,2,2025-02-24 15:00:00,2025-02-24 17:00:00
...,...,...,...
933,2,2025-02-28 10:36:00,2025-02-28 13:01:00
934,4,2025-02-28 13:01:00,2025-02-28 13:56:00
935,5,2025-02-28 13:56:00,2025-02-28 15:25:00
936,7,2025-02-28 15:25:00,2025-02-28 17:05:00


In [14]:
df.query("FinEvento == '·'")

,Item,Cuenta,Evento,Actividad,Alimentador,Primario,Desconexion,SIG,Tipo,Materiales,...,Duracion,Responsable,Colaboradores,HorasExtra,Vehiculo,Sitio,id_ot,Archivo,uuid,anio_mes
539,10,MEDIDORES,Aprovechando que se esta en el sitio se instal...,PROG,Paquisha,No,No,No,EXPANSION,·,...,-1,OCHOA JARAMILLO ANGEL CLAUDIO,2,No,4-62,Yantzaza - Paquisha - Bellavista.,148321,Orden de trabajo Paquisha 28-01-2025 (AO).pdf,5e089466-ab6a-49c1-9f48-095574963046,2025-02-28T00:00:00-05


In [ ]:
[print( file ) for file in list_pdfs]

In [ ]:
[print( f" Ot Link: {ot.log} ") for ot in obj_lists]

### Generar la Matriz de Actividades para un objeto

In [8]:
nro_ot = 0
obj_lists[nro_ot].data["log"]

[{'t': '2025-03-25T00:37:38.203106',
  'level': 'INFO',
  'message': 'Se encuentra un archivo PDF de al menos tres hojas ',
  'detail': 'Ninguno'},
 {'t': '2025-03-25T00:37:38.214969',
  'level': 'ERROR',
  'message': 'No coinciden la Fecha de la OT viernes, 11 de febrero del 2022 con Fecha de Inicio: sábado, 12 de febrero del 2022',
  'detail': '|>> Comparando las dos fechas de Hoja Uno <<|'}]

In [7]:
test = obj_lists[nro_ot].load_ot()
actividades = pd.DataFrame(test.data["actividades"])

In [7]:
fechaModa = test.data['fecha']
type(fechaModa)

str

In [18]:
matriz_test = matrizActividades.ConvertirOT_a_ActividadesCSV(  obj_lists[nro_ot] )
#matriz_test[['Cuenta','Evento','Fecha','InicioEvento','FinEvento']]
#matriz_test[['Fecha','InicioEvento','corregir_fechaInicio','FinEvento','corregir_fechaFin']]
matriz_test[['Fecha','InicioEvento','FinEvento']]

/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/.venv/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MultinomialNB from version 1.4.1.post1 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/.venv/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator CountVectorizer from version 1.4.1.post1 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


,Fecha,InicioEvento,FinEvento
0,2024-05-22 00:00:00,2024-05-22 08:00:00,2024-05-22 08:22:00
2,2024-05-22 00:00:00,2024-05-23 08:22:00,2024-05-23 08:33:00
3,2024-05-22 00:00:00,2024-05-22 08:53:00,2024-05-22 09:28:00
5,2024-05-22 00:00:00,2024-05-22 09:28:00,2024-05-22 09:59:00
7,2024-05-22 00:00:00,2024-05-22 09:59:00,2024-05-22 10:08:00
8,2024-05-22 00:00:00,2024-05-22 10:08:00,2024-05-22 10:39:00
9,2024-05-22 00:00:00,2024-05-22 10:39:00,2024-05-22 11:35:00
10,2024-05-22 00:00:00,2024-05-22 11:35:00,2024-05-22 12:30:00
11,2024-05-22 00:00:00,2024-05-22 12:30:00,2024-05-22 13:30:00
12,2024-05-22 00:00:00,2024-05-22 13:30:00,2024-05-22 14:18:00


In [15]:
dbg

version                                                     0.12.0
link             /home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/te...
id_ot                                                     134508.0
exito                                                         True
cuadrilla                          Yacuambi Z1 (Cuadrilla. Nro. 8)
responsable                     [LOZANO SIGCHO NAUN ENRIQUE, JECE]
colaboradores    {'total': 3, 'nombres': [['CABRERA GONZALEZ LU...
diaSemana                                                    lunes
fecha                                    2024-07-29 00:00:00-05:00
fechaInicio                            lunes, 29 de julio del 2024
fechaFinal                                     29/07/2024 20:40:00
sitio                 Yacuambi - Tamboloma, Hucapamba y Jembuentza
descripcion      Traslado a Tamboloma para revisar sector sin s...
tEstimado                                                        8
vehiculo         {'numero': 'R-171', 'placa': 'AAA-4278', 'mar

In [16]:
test.data['fechaFinal']

'11/02/2022 23:00:00'

In [26]:
from datetime import datetime
from pytz import timezone

fechaFinal = test.data['fechaFinal']

datetime_object = datetime.strptime(fechaFinal, '%d/%m/%Y %H:%M:%S')
ecuador = timezone("America/Guayaquil")
local_datetime = ecuador.localize(datetime_object)
fechaFinal = local_datetime.isoformat()
solofechaFinal = fechaFinal.split('T')[1]
solofechaFinal

'23:00:00-05:00'

### Secuencial GLOBAL LOCK

In [ ]:
### Secuencial en un solo procesador.
obj_lists = []
for file in list_pdfs:
  ot = gestionOT.GestionOt( file )
  ot.load_ot()
  obj_lists.append( ot )

## TEST TEST

In [2]:
date = "2025-03-05 09:00:00"